# 2 - Signature, enrichment and figures

From the nine per-dataset tables of notebook 1 to the consensus signature (Sections
4.3-4.4), its over-representation analysis (Section 4.5) and the three figures.

| Output | Use |
|---|---|
| `results/Table2_signature.csv` | Table 2: signature genes, with direction and statistics |
| `results/TableS1_enrichment.csv` | Table S1: enriched terms |
| `results/Table3_pathways.csv` | Table 3: KEGG and Reactome pathways with FDR < 1e-4 |
| `figures/Figure1_dataset_selection.png`, `figures/Figure2_interventions.png`, `figures/Figure3_workflow.png` | Figures 1-3 |
| `results/orthologs_per_dataset.csv`, `results/signature_one_to_one_sensitivity.csv` | sensitivity analysis on one-to-one orthologs |

In [1]:
# Move to the project root (the folder that contains run_all.ps1), so that relative
# paths work whether the notebook is started from notebook/ or from the root.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "run_all.ps1")) && dirname(root) != root) {
  root <- dirname(root)
}
if (!file.exists(file.path(root, "run_all.ps1"))) {
  stop("project root not found: no run_all.ps1 above ", getwd())
}
setwd(root)
.libPaths(c(file.path(root, "env", "rlib"), .libPaths()))
Sys.setenv(RENAL_IRI_ROOT = root)
options(stringsAsFactors = FALSE, warn = 1)
cat(R.version.string, "\nroot: ", root, "\n", sep = "")

R version 4.4.3 (2025-02-28)
root: /Users/robertocasale/Desktop/lavoro_FBF/renal_shield_perfusion


## Setup

Shared settings (`R/helpers.R`), ortholog mapping and rank aggregation
(`R/pipeline_meta.R`), the signature rule (`R/pipeline_signature.R`) and the enrichment
(`R/pipeline_ora.R`).

In [2]:
source("R/helpers.R")
source("R/pipeline_meta.R")
source("R/pipeline_signature.R")
source("R/pipeline_ora.R")

[2026-10-07 23:35:12] INFO  helpers.R loaded; constants and robust_fetch() ready.



## Human orthologs

Every dataset is expressed in human gene symbols (Section 4.3).

In [3]:
fs <- list.files("results", "^DE_GSE.*[.]csv$", full.names = TRUE)
stopifnot(length(fs) == 9)
human_de <- list()
for (f in fs) {
  d <- read.csv(f)
  human_de[[unique(d$accession)[1]]] <- collapse_human(map_to_human(d, unique(d$organism)[1]))
}
cat("datasets:", paste(names(human_de), collapse = ", "), "\n")
cat("human genes per dataset:", paste(sapply(human_de, nrow), collapse = ", "), "\n")

[2026-10-07 23:35:49] INFO    Rattus norvegicus: 3293 outdated rodent symbols updated before ortholog mapping



[2026-10-07 23:35:56] INFO    Rattus norvegicus: 6106 outdated rodent symbols updated before ortholog mapping



[2026-10-07 23:36:05] INFO    Mus musculus: 1431 outdated rodent symbols updated before ortholog mapping



[2026-10-07 23:36:07] INFO    Mus musculus: 1763 outdated rodent symbols updated before ortholog mapping



[2026-10-07 23:36:09] INFO    Rattus norvegicus: 3917 outdated rodent symbols updated before ortholog mapping



[2026-10-07 23:36:12] INFO    Mus musculus: 1919 outdated rodent symbols updated before ortholog mapping



datasets: GSE126805, GSE148420, GSE27274, GSE30718, GSE34351, GSE39548, GSE43974, GSE58438, GSE98622 


human genes per dataset: 16854, 15216, 12862, 22839, 16123, 12784, 34212, 15165, 16852 


## Signature

Rank aggregation in each direction across the nine datasets, then the signature rule
(Section 4.4): genes measured in at least 5 of the 9 datasets, Benjamini-Hochberg adjusted
p < 0.05 in the assigned direction, and the same direction in at least half of the
datasets that measure the gene.

In [4]:
cons <- build_consensus(human_de)
sig  <- posthoc_signature(cons)
save_csv(sig[sig$signature, ], "Table2_signature.csv")
cat("consensus signature:", sum(sig$signature), "genes\n")
print(head(sig[sig$signature, c("human","mean_logFC","FDR_posthoc","concordant","n_datasets")], 12),
      row.names = FALSE, digits = 3)

[2026-10-07 23:36:20] INFO  Post-hoc signature: 256 genes (16540 tested at coverage >= 5)



[2026-10-07 23:36:20] INFO  WROTE       results/Table2_signature.csv (256 rows)



consensus signature: 256 genes


     human mean_logFC FDR_posthoc concordant n_datasets
   ADAMTS1       2.34    5.15e-09          9          9
       MYC       2.05    1.15e-07          9          9
      KLF6       2.33    5.07e-07          9          9
 TNFRSF12A       2.28    5.07e-07          9          9
     CLDN4       1.93    5.07e-07          9          9
     TGIF1       1.54    5.07e-07          9          9
     GDF15       2.25    1.25e-06          9          9
    NFKBIZ       1.93    1.25e-06          9          9
  SERPINE1       2.26    1.25e-06          9          9
      ELF3       1.26    1.94e-06          9          9
     ICAM1       1.37    2.07e-06          9          9
     SOCS3       2.12    2.07e-06          9          9


## Sensitivity analysis - one-to-one orthologs only

A rodent gene enters the analysis through its human ortholog. Most genes have exactly one.
Some belong to gene families that expanded differently in rodents and humans (for example
the CXCL chemokines); for these the main analysis uses the match supported by the most
orthology databases. This section counts the orthologs in each rodent dataset, then
removes every gene without a one-to-one ortholog and derives the signature again with the
same functions as above. The three human datasets are not changed.

A rodent-human pair is one-to-one when, among all the pairs that babelgene accepts (those
supported by at least three orthology databases), the rodent gene matches only that human
gene and the human gene matches only that rodent gene.

In [5]:
# 1. The one-to-one orthologs of each rodent species, from every pair that babelgene
#    accepts. top = FALSE returns all the matches of a gene, not only the best one, so a
#    gene with more than one match can be recognised.
species <- c("Mus musculus" = "mouse", "Rattus norvegicus" = "rat")
human_ids <- AnnotationDbi::keys(org.Hs.eg.db::org.Hs.eg.db, keytype = "ENTREZID")
one_to_one <- list()
for (org in names(species)) {
  p <- babelgene::orthologs(genes = human_ids, species = species[[org]],
                            human = TRUE, top = FALSE)
  p$human <- harmonise_human_symbol(p$human_symbol)   # the gene symbols the analysis uses
  n_rodent <- tapply(p$symbol, p$human, function(z) length(unique(z)))
  n_human  <- tapply(p$human, p$symbol, function(z) length(unique(z)))
  one_to_one[[org]] <- unique(p$human[n_rodent[p$human] == 1 & n_human[p$symbol] == 1])
}
# two genes with a known answer: MYC has a single ortholog in mouse and in rat, while
# CXCL2 belongs to the CXCL1/CXCL2/CXCL3 family, which has none
stopifnot(all(sapply(one_to_one, function(g) "MYC" %in% g)),
          !any(sapply(one_to_one, function(g) "CXCL2" %in% g)))

# 2. For each rodent dataset: the rodent genes measured, how many have a human ortholog
#    (the others are excluded by the mapping above), and how many of the resulting human
#    genes have a one-to-one ortholog.
orthologs_per_dataset <- do.call(rbind, lapply(fs, function(f) {
  d <- read.csv(f, stringsAsFactors = FALSE)
  org <- unique(d$organism)[1]
  if (!org %in% names(species)) return(NULL)
  acc <- unique(d$accession)[1]
  m <- suppressMessages(map_to_human(d, org))        # the same mapping as above
  m <- m[!is.na(m$gene) & !is.na(m$human) & m$human != "", ]
  rodent <- unique(d$gene[!is.na(d$gene) & d$gene != ""])
  data.frame(dataset = acc, species = species[[org]],
             rodent_genes = length(rodent),
             with_ortholog = length(unique(m$gene)),
             pct = round(100 * length(unique(m$gene)) / length(rodent), 1),
             human_genes = nrow(human_de[[acc]]),
             one_to_one = sum(human_de[[acc]]$human %in% one_to_one[[org]]))
}))
save_csv(orthologs_per_dataset, "orthologs_per_dataset.csv")
print(orthologs_per_dataset, row.names = FALSE)

[2026-10-07 23:36:36] INFO  WROTE       results/orthologs_per_dataset.csv (6 rows)



   dataset species rodent_genes with_ortholog  pct human_genes one_to_one
 GSE148420     rat        19297         15648 81.1       15216      14208
  GSE27274     rat        21791         13353 61.3       12862      12022
  GSE34351   mouse        21495         16458 76.6       16123      15096
  GSE39548   mouse        14330         12978 90.6       12784      11968
  GSE58438     rat        18784         15570 82.9       15165      14167
  GSE98622   mouse        29325         17451 59.5       16852      15661


In [6]:
# 3. The signature derived again from one-to-one orthologs only. Each rodent dataset keeps
#    only its one-to-one genes; the human datasets are unchanged. The derivation is the one
#    used above (build_consensus, posthoc_signature): only the input differs.
human_de_1to1 <- human_de
for (acc in orthologs_per_dataset$dataset) {
  org <- names(species)[species == orthologs_per_dataset$species[orthologs_per_dataset$dataset == acc]]
  human_de_1to1[[acc]] <- human_de[[acc]][human_de[[acc]]$human %in% one_to_one[[org]], ]
}
sig_1to1 <- posthoc_signature(build_consensus(human_de_1to1))

# A gene is tested only if it is measured in at least 5 of the 9 datasets. Some signature
# genes keep fewer once their rodent values are removed, so this run cannot test them;
# they are counted apart from the genes that are tested and not found again.
main     <- sig$human[sig$signature]
alt      <- sig_1to1$human[sig_1to1$signature]
testable <- main[main %in% sig_1to1$human]
found    <- intersect(testable, alt)
same_direction <- all(sig$direction[match(found, sig$human)] ==
                      sig_1to1$direction[match(found, sig_1to1$human)])
cat(sprintf("signature genes of the main analysis: %d\n", length(main)))
cat(sprintf("  not testable without rodent values (fewer than 5 datasets): %d\n    %s\n",
            length(setdiff(main, testable)), paste(sort(setdiff(main, testable)), collapse = ", ")))
cat(sprintf("  testable: %d, of which found again: %d (%.1f%%), same direction: %s\n",
            length(testable), length(found), 100 * length(found) / length(testable), same_direction))
cat(sprintf("  testable and not found again: %d\n    %s\n",
            length(setdiff(testable, alt)), paste(sort(setdiff(testable, alt)), collapse = ", ")))
cat(sprintf("new genes in the one-to-one signature: %d\n    %s\n",
            length(setdiff(alt, main)), paste(sort(setdiff(alt, main)), collapse = ", ")))
both <- union(main, alt)
save_csv(data.frame(human = both,
                    main_signature = both %in% main,
                    one_to_one_signature = both %in% alt,
                    testable_one_to_one = both %in% sig_1to1$human,
                    FDR_main = sig$FDR_posthoc[match(both, sig$human)],
                    FDR_one_to_one = sig_1to1$FDR_posthoc[match(both, sig_1to1$human)]),
         "signature_one_to_one_sensitivity.csv")

[2026-10-07 23:36:42] INFO  Post-hoc signature: 234 genes (15675 tested at coverage >= 5)



signature genes of the main analysis: 256


  not testable without rodent values (fewer than 5 datasets): 14
    AKR1C2, ANXA2, BTG1, CCL2, CXCL2, CYP4A22, HSPA1A, MT1F, MT2A, PLSCR1, SERPINA3, TUBA1C, TUBB2A, TUBB2B


  testable: 242, of which found again: 232 (95.9%), same direction: TRUE


  testable and not found again: 10
    CLIC1, HMGA1, MYL12A, S100A11, SFN, SLC19A2, SLC25A25, SLCO4C1, SRXN1, TBRG1


new genes in the one-to-one signature: 2
    MYH9, TAX1BP3


[2026-10-07 23:36:42] INFO  WROTE       results/signature_one_to_one_sensitivity.csv (258 rows)



## Enrichment

WebGestalt over-representation analysis of the signature against the tested genes
(Section 4.5).

In [7]:
# The background is the set of tested genes (measured in at least 5 of the 9 datasets).
universe <- sig$human
cat("interest:", sum(sig$signature), "genes | background:", length(universe), "tested genes\n")
ora <- run_ora_set(sig$human[sig$signature], "signature", background = universe)
stopifnot(nrow(ora) > 0)
save_csv(ora, "TableS1_enrichment.csv")
cat(nrow(ora), "enriched terms\n")
print(table(ora$database))
# Table 3: KEGG and Reactome pathways with FDR < 1e-4.
table3 <- ora[grepl("^pathway", ora$database) & ora$FDR < 1e-4,
              c("database", "geneSet", "description", "size", "overlap", "enrichmentRatio", "FDR")]
save_csv(table3, "Table3_pathways.csv")
print(table3[, c("database", "description", "overlap", "enrichmentRatio", "FDR")], row.names = FALSE, digits = 3)

interest: 256 genes | background: 16540 tested genes


[2026-10-07 23:36:42] INFO  CACHE HIT   webgestalt_ora_signature_pathway_KEGG_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:37)



[2026-10-07 23:36:42] INFO  CACHE HIT   webgestalt_ora_signature_pathway_Reactome_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:44)



[2026-10-07 23:36:42] INFO  CACHE HIT   webgestalt_ora_signature_drug_DrugBank_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:50)



[2026-10-07 23:36:43] INFO  CACHE HIT   webgestalt_ora_signature_drug_GLAD4U_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:58)



[2026-10-07 23:36:43] INFO  WROTE       results/TableS1_enrichment.csv (134 rows)



134 enriched terms



   drug_DrugBank      drug_GLAD4U     pathway_KEGG pathway_Reactome 
               1               59               26               48 


[2026-10-07 23:36:43] INFO  WROTE       results/Table3_pathways.csv (13 rows)



         database                                               description
     pathway_KEGG                                     TNF signaling pathway
     pathway_KEGG                                   IL-17 signaling pathway
 pathway_Reactome                       Cytokine Signaling in Immune system
 pathway_Reactome                Interleukin-4 and Interleukin-13 signaling
 pathway_Reactome                                 Signaling by Interleukins
 pathway_Reactome                                  Interleukin-10 signaling
     pathway_KEGG                                    MAPK signaling pathway
 pathway_Reactome                                             Immune System
     pathway_KEGG                                                 Apoptosis
     pathway_KEGG                              NF-kappa B signaling pathway
     pathway_KEGG                                Osteoclast differentiation
 pathway_Reactome HSP90 chaperone cycle for steroid hormone receptors (SHR)
 pathway_Rea

## Figures

Figure 1 (dataset selection), Figure 2 (pathways and candidate interventions; the code
stops if a pathway or a gene drawn as part of the signature is no longer supported by the
results) and Figure 3 (workflow).

In [8]:
source("R/figures.R")
dir.create("figures", showWarnings = FALSE)
ident <- read.csv("results/geo_identification.csv", stringsAsFactors = FALSE)
cover <- if (file.exists("results/geo_corpus_coverage.csv"))
  read.csv("results/geo_corpus_coverage.csv", stringsAsFactors = FALSE) else NULL
invisible(fig_dataset_prisma(
  ident$n_identified[1], length(human_de), "figures/Figure1_dataset_selection.png",
  organisms = vapply(human_de, function(d) d$organism[1], character(1)),
  searched_on = ident$searched_on[1], coverage = cover, query = ident$query[1]))
common_name <- c("Homo sapiens" = "human", "Mus musculus" = "mouse", "Rattus norvegicus" = "rat")
orgs <- vapply(human_de, function(d) d$organism[1], character(1))
tb   <- table(ifelse(orgs %in% names(common_name), common_name[orgs], orgs))
invisible(fig_interventions("figures/Figure2_interventions.png", ora = ora, sig = sig))
invisible(fig_workflow("figures/Figure3_workflow.png",
  n_sig   = sum(sig$signature),
  n_terms = nrow(ora),
  n_path  = sum(grepl('^pathway', ora$database)),
  n_drug  = sum(grepl('^drug', ora$database)),
  n_datasets  = length(human_de),
  composition = paste(sprintf("%d %s", as.integer(tb), names(tb)), collapse = "   |   ") ))
cat("wrote figures/Figure1_dataset_selection.png, figures/Figure2_interventions.png and figures/Figure3_workflow.png
")

wrote figures/Figure1_dataset_selection.png, figures/Figure2_interventions.png and figures/Figure3_workflow.png


Next: `03_validation.ipynb`.